In [32]:
import numpy as np

from datasets import load_dataset
from transformers import AutoTokenizer,AutoModelForQuestionAnswering, TrainingArguments,Trainer,DefaultDataCollator

import evaluate
from sklearn.metrics import accuracy_score, f1_score
import torch

In [33]:
dataset = load_dataset("squad")

dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

In [34]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-uncased")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [35]:
def preprocess_function(examples):

    questions = [q.strip() for q in examples["question"]]

    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=512,
        truncation="only_second",
        stride=256,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs["offset_mapping"]
    sample_map = inputs["overflow_to_sample_mapping"]

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):

        sample_idx = sample_map[i]

        answers = examples["answers"][sample_idx]

        if len(answers["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)

        else:
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            sequence_ids = inputs.sequence_ids(i)

            idx = 0
            while sequence_ids[idx] != 1:
                idx += 1

            context_start = idx

            while sequence_ids[idx] == 1:
                idx += 1

            context_end = idx - 1

            if (
                offsets[context_start][0] > end_char
                or offsets[context_end][1] < start_char
            ):
                start_positions.append(0)
                end_positions.append(0)

            else:

                idx = context_start

                while (
                    idx <= context_end
                    and offsets[idx][0] <= start_char
                ):
                    idx += 1

                start_positions.append(idx - 1)

                idx = context_end

                while (
                    idx >= context_start
                    and offsets[idx][1] >= end_char
                ):
                    idx -= 1

                end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    return inputs

In [36]:
tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'overflow_to_sample_mapping', 'start_positions', 'end_positions'],
        num_rows: 87724
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'overflow_to_sample_mapping', 'start_positions', 'end_positions'],
        num_rows: 10630
    })
})

In [37]:
data_collator = DefaultDataCollator()

In [38]:
metric = evaluate.load("squad")

def compute_metrics(eval_pred):
    return {}

In [39]:
training_args = TrainingArguments(
    output_dir="./qa_model",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
)

In [40]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator
)
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.186813,1.121755
2,0.724972,1.156126


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=21932, training_loss=1.092181531091078, metrics={'train_runtime': 2774.6883, 'train_samples_per_second': 63.232, 'train_steps_per_second': 7.904, 'total_flos': 2.2922825094414336e+16, 'train_loss': 1.092181531091078, 'epoch': 2.0})

In [ ]:
results = trainer.evaluate()

results

with open("evaluation_results.txt", "w") as f:
    for key, value in results.items():
        f.write(f"{key}: {value}\n")

In [42]:
trainer.save_model("fine_tuned_qa_model")

tokenizer.save_pretrained("fine_tuned_qa_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('fine_tuned_qa_model/tokenizer_config.json',
 'fine_tuned_qa_model/tokenizer.json')

In [ ]:
from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model="fine_tuned_qa_model",
    tokenizer="fine_tuned_qa_model"
)

context = """
Transformers are deep learning models developed by researchers.
They are widely used in NLP tasks.
"""

question = "What are transformers used for?"

result = qa_pipeline(
    question=question,
    context=context
)

result

{'score': 0.5089776515960693, 'start': 89, 'end': 98, 'answer': 'NLP tasks'}